# Ch. 6 · Ajustement d’un modèle physique et incertitude

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch06-modele-scientifique.ipynb)

**IA pour les sciences — M1.** Loi de décroissance : problème inverse, incertitude des paramètres, identifiabilité et extrapolation.

**Durée indicative :** 1 h 30 (atelier facultatif en autonomie)  
**Statut :** atelier facultatif hors volume horaire.

**Objectifs**

- Estimer les paramètres d’une loi exponentielle par moindres carrés non linéaires.
- Tirer de la matrice de covariance une incertitude et un intervalle sur k et sur la demi-vie.
- Comparer à la linéarisation logarithmique et en comprendre les limites.
- Montrer qu’un modèle mécaniste extrapole mieux qu’un modèle flexible… s’il est juste.
- Mettre en évidence un problème mal posé (paramètres non identifiables).

**Prérequis :** chapitre 3 (régression) et cours du chapitre 6.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 1. Observations simulées et problème inverse

Loi : $C(t) = C_0 e^{-kt}$. **Problème direct** : connaissant $(C_0, k)$, calculer $C(t)$. **Problème inverse** : à partir de mesures
bruitées, estimer $(\hat C_0, \hat k) = \arg\min \sum_i [C_i - C_0e^{-kt_i}]^2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

rng = np.random.default_rng(42)
t = np.linspace(0, 6, 21)
C0, k_reel, sigma = 8.0, 0.4, 0.15
C_mesure = C0 * np.exp(-k_reel * t) + rng.normal(0, sigma, len(t))

def loi(t, c0, k):
    return c0 * np.exp(-k * t)

params, cov = curve_fit(loi, t, C_mesure, p0=[5, 0.1], bounds=([0, 0], [np.inf, np.inf]))
erreurs = np.sqrt(np.diag(cov))
for nom, v, e, vrai in zip(['C0', 'k'], params, erreurs, [C0, k_reel]):
    print(f'{nom} = {v:.4f} ± {1.96*e:.4f} (IC 95 % approx.) — valeur simulée {vrai}')

**Lire l’incertitude.** `curve_fit` renvoie la matrice de covariance des paramètres (approximation linéaire autour de l’optimum).
L’écart-type de $\hat k$ est $\sqrt{\text{cov}_{kk}}$ et un intervalle à 95 % vaut environ $\hat k \pm 1{,}96\,\sigma_{\hat k}$.
La **demi-vie** $t_{1/2} = \ln 2 / k$ hérite de cette incertitude (propagation : $\sigma_{t_{1/2}} \approx \frac{\ln 2}{k^2}\sigma_k$).

In [ ]:
k_hat, s_k = params[1], erreurs[1]
t_demi = np.log(2) / k_hat
s_t = np.log(2) / k_hat**2 * s_k
print(f'Demi-vie estimée : {t_demi:.3f} ± {1.96*s_t:.3f} (vraie : {np.log(2)/k_reel:.3f})')

residus = C_mesure - loi(t, *params)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
grille = np.linspace(0, 6, 200)
axes[0].scatter(t, C_mesure, label='Mesures simulées')
axes[0].plot(grille, loi(grille, *params), label='Ajustement')
axes[0].set(xlabel='Temps (h)', ylabel='Concentration (mg/L)', title='Loi ajustée'); axes[0].legend()
axes[1].scatter(t, residus); axes[1].axhline(0, color='k', ls='--')
axes[1].set(xlabel='Temps (h)', ylabel='Résidu (mg/L)', title='Résidus : sans structure ?')
plt.tight_layout(); plt.show()

## 2. Démonstration : la linéarisation logarithmique

$\ln C = \ln C_0 - kt$ : une simple régression linéaire de $\ln C$ sur $t$ donne $k$. C’est tentant… mais le bruit additif
devient **hétéroscédastique** après le logarithme : les faibles concentrations, très bruitées en relatif, pèsent autant que les autres.
Et une mesure négative rend le logarithme impossible.

In [ ]:
positif = C_mesure > 0
pente, origine = np.polyfit(t[positif], np.log(C_mesure[positif]), 1)
print(f'k par linéarisation : {-pente:.4f}   |   k non linéaire : {k_hat:.4f}   |   vrai : {k_reel}')

# répéter l'expérience 300 fois pour comparer les deux estimateurs
k_lin, k_nl = [], []
for _ in range(300):
    Cm = C0 * np.exp(-k_reel * t) + rng.normal(0, 0.3, len(t))
    m = Cm > 0
    k_lin.append(-np.polyfit(t[m], np.log(Cm[m]), 1)[0])
    k_nl.append(curve_fit(loi, t, Cm, p0=[5, .1], bounds=([0, 0], [np.inf, np.inf]))[0][1])
plt.hist(k_lin, bins=30, alpha=.6, label=f'linéarisation (moy {np.mean(k_lin):.3f})')
plt.hist(k_nl, bins=30, alpha=.6, label=f'non linéaire (moy {np.mean(k_nl):.3f})')
plt.axvline(k_reel, color='k', ls='--', label='vraie valeur')
plt.xlabel('k estimé'); plt.legend(); plt.title('300 expériences simulées (bruit 0,3)'); plt.show()

## 3. Démonstration : extrapoler avec une loi ou avec un modèle flexible

On n’observe que $t\in[0, 3]$ h. Un polynôme de degré 4 s’ajuste aussi bien, voire mieux, sur ce domaine…
mais extrapole n’importe comment. Le modèle mécaniste extrapole bien **parce que la loi est juste** : si la cinétique change au-delà de 3 h,
il se trompera aussi.

In [ ]:
court = t <= 3
p_exp, _ = curve_fit(loi, t[court], C_mesure[court], p0=[5, .1], bounds=([0, 0], [np.inf, np.inf]))
p_poly = np.polyfit(t[court], C_mesure[court], 4)
grille = np.linspace(0, 8, 300)
plt.scatter(t[court], C_mesure[court], label='Observé (t ≤ 3 h)')
plt.scatter(t[~court], C_mesure[~court], color='lightgray', label='Non observé')
plt.plot(grille, loi(grille, *p_exp), label='Loi exponentielle')
plt.plot(grille, np.polyval(p_poly, grille), label='Polynôme degré 4')
plt.axvspan(3, 8, color='gray', alpha=.08); plt.ylim(-4, 10)
plt.xlabel('Temps (h)'); plt.ylabel('Concentration (mg/L)'); plt.legend(); plt.title('Interpolation ≠ extrapolation'); plt.show()

### ✏️ Exercice 1 — Identifiabilité : une fenêtre de mesure trop courte

Refaites l’ajustement en n’utilisant que les mesures avec $t \le 0{,}9$ h (4 points), puis avec $t \le 1{,}5$ h. Comparez l’intervalle à 95 % de $k$. Que conclure sur le **plan d’expérience** ?

In [ ]:
for t_max in [0.9, 1.5, 6]:
    m = t <= t_max
    p, c = curve_fit(loi, t[m], C_mesure[m], p0=[5, .1], bounds=([0, 0], [np.inf, np.inf]))
    s = ...
    print(f't ≤ {t_max} h ({m.sum()} points) : k = {p[1]:.3f} ± {1.96*s:.3f}')

<details><summary><b>Voir le corrigé</b></summary>

```python
for t_max in [0.9, 1.5, 6]:
    m = t <= t_max
    p, c = curve_fit(loi, t[m], C_mesure[m], p0=[5, .1], bounds=([0, 0], [np.inf, np.inf]))
    s = np.sqrt(c[1, 1])
    print(f't ≤ {t_max} h ({m.sum()} points) : k = {p[1]:.3f} ± {1.96*s:.3f}')
```

Sur une fenêtre courte, la décroissance ressemble à une droite : $C_0$ et $k$ se compensent et l’intervalle sur $k$ devient environ cinq fois plus large. Le paramètre est mal **identifiable**. Mieux vaut mesurer sur plusieurs demi-vies : le plan d’expérience détermine la précision atteignable.

</details>

### ✏️ Exercice 2 — Intervalle par bootstrap des résidus

Estimez un intervalle à 95 % de $k$ par **bootstrap** : 500 fois, créez $C^* = \hat C(t) + $ résidus tirés avec remise, réajustez, puis prenez les percentiles 2,5 et 97,5 %. Comparez à l’intervalle de `curve_fit`.

In [ ]:
ajuste = loi(t, *params)
k_boot = []
for _ in range(500):
    C_star = ajuste + rng.choice(residus, size=len(t), replace=True)
    p, _ = curve_fit(loi, t, C_star, p0=params, bounds=([0, 0], [np.inf, np.inf]))
    k_boot.append(...)
bas, haut = np.percentile(k_boot, [2.5, 97.5])
print(f'IC bootstrap : [{bas:.4f}, {haut:.4f}]   IC curve_fit : [{k_hat-1.96*s_k:.4f}, {k_hat+1.96*s_k:.4f}]')

<details><summary><b>Voir le corrigé</b></summary>

```python
ajuste = loi(t, *params)
k_boot = []
for _ in range(500):
    C_star = ajuste + rng.choice(residus, size=len(t), replace=True)
    p, _ = curve_fit(loi, t, C_star, p0=params, bounds=([0, 0], [np.inf, np.inf]))
    k_boot.append(p[1])
bas, haut = np.percentile(k_boot, [2.5, 97.5])
print(f'IC bootstrap : [{bas:.4f}, {haut:.4f}]   IC curve_fit : [{k_hat-1.96*s_k:.4f}, {k_hat+1.96*s_k:.4f}]')
```

Les deux intervalles sont proches ici, car le modèle est correct et le bruit gaussien. Le bootstrap ne repose pas sur l’approximation linéaire : il reste utilisable quand celle-ci est douteuse.

</details>

## Interprétation scientifique (à rédiger)

1. Les résidus de l’ajustement sont-ils compatibles avec un bruit de mesure pur ?
2. Quelle méthode d’estimation de k recommandez-vous, et pourquoi ?
3. Dans quelles conditions accepteriez-vous d’extrapoler la loi au-delà de la fenêtre mesurée ?
4. Quel plan de mesure (instants, répétitions) proposeriez-vous pour estimer k à ± 2 % près ?